# Hybrid Search — Dense & Sparse Retrieval

## ① Sparse Retrieval = keyword matching
- Matches **exact words** using old-school NLP: **TF-IDF, BM25, BoW** (bag of words).
- Turns text into a **sparse matrix** — mostly zeros, a `1` where a word exists.
- Example from the notes: `"I want to have food"` → `[1, 0, 0, 0, 1]` (1s mark which known words appear).
- Great at **exact match**, dumb about meaning. Searching "car" won't find "automobile."

## ② Dense Retrieval = meaning matching
- Uses **vector embeddings + cosine similarity** (yep, exactly what you built with MiniLM).
- Stored in **FAISS / Chroma**.
- Finds **semantically similar** sentences even if the words differ. "car" *does* find "automobile."
- Great at meaning, but can miss when you need a **precise keyword** (like a product code or a name).

## Why combine them → Hybrid
Each one has a blind spot, so you just... use both.
> Sparse gives you **exact keyword match**, dense gives you **semantic power** — hybrid = best of both worlds.

## The formula — this is the money part

Where:
- **Score_dense** = cosine similarity (input vs vector store)
- **Score_sparse** = TF-IDF score (input vs keyword representation)
- **α (alpha)** = the weight knob, here `α = 0.5`

So α is just "how much do I trust meaning vs keywords." `0.5` = trust both equally. Crank α up → lean semantic; crank it down → lean keyword. Same **dial** vibe as `temperature`, just for blending two scores.

## The worked example — watch it click
Query: **"build application using LLM"**. Each doc gets two scores:

| Doc | Dense (cosine) | Sparse (TF-IDF) | Hybrid = 0.5×dense + 0.5×sparse |
|-----|----------------|-----------------|---------------------------------|
| D1 "LangChain helps build LLM apps" | 0.85 | 0.60 | **≈ 0.82** ✅ |
| D2 "Pinecone for vector search"     | 0.40 | 0.20 | ≈ 0.18 |
| D3 "Eiffel Tower is in Paris"       | 0.10 | 0.10 | ≈ 0.04 |

*(the exact decimals in the notes are a little rough, but the ranking is the point)*

**D1 wins big** because it scores high on **both** — it's semantically about building LLM apps AND literally contains the keywords "build," "LLM," "apps." D3 (Eiffel Tower) bombs both, as it should. That's hybrid doing its job: a doc that's strong on both signals rises to the top.

## TL;DR
- **Sparse** = exact words (TF-IDF/BM25), **Dense** = meaning (embeddings + cosine).
- **Hybrid** blends their scores with a weight `α`.
- `Score_hybrid = α·dense + (1−α)·sparse`, α=0.5 means equal trust.
- You already know both halves — this note just teaches you to **fuse** them so retrieval is both **smart and precise**.

# Re-Ranking Techniques (Hybrid Search Strategies)

## The core idea
Retrieval is **two-stage** now:
1. **Fast retriever** (BM25 / FAISS / hybrid) grabs top-k docs quickly — cheap but rough.
2. **Slower, smarter model** (cross-encoder or LLM) **re-scores and reorders** those k docs by actual relevance to the query.

> Get candidates fast → then carefully re-sort them so the best one lands on top.

## Why not just use the accurate model directly?
Because it's expensive. You can't run a cross-encoder over 100k docs. So: **cheap retriever narrows 100k → 10, then the expensive model perfectly orders those 10.** Best of both.


## The 3 stages

Re-ranking is the **new middle layer** you're inserting between what you already built and the LLM.

## Why it matters
- **Relevance** — top-k from a retriever is often only *loosely* related; reranker fixes the order.
- **Less hallucination** — irrelevant docs get filtered → grounded answers.
- **Query intent** — cross-encoder reads the **query+doc together**, so it actually understands intent (a bi-encoder embeds them separately, which loses nuance).
- **Noise reduction** — junk chunks get pushed to the bottom.

## The key technical bit
- **Bi-encoder (retrieval):** embeds query and doc *separately* → compare vectors. Fast, but shallow.
- **Cross-encoder (reranking):** feeds query **and** doc together into the model → outputs a relevance score. Way more accurate, way slower — which is exactly why it only runs on the top-k.

## TL;DR
> Retriever = fast but sloppy. Reranker = slow but precise. Use the retriever to shortlist, the reranker to sort. Result: the LLM gets the genuinely best context → better answers.

                      Input Query
                           │
                           ▼
                    ┌─────────────┐
                    │ Vectorstore │
                    └─────────────┘
                     ╱           ╲
               Exact              Similar
                  ╱                   ╲
          ┌──────────┐        ┌──────────────────┐
          │  BM25    │        │ Semantic Search  │ → FAISS
          │          │        │   Embedding      │
          └──────────┘        └──────────────────┘
                │                       │
              Top K                   Top K
                │                       │
                └────────┬──────────────┘
                         ▼
                 ┌───────────────┐
                 │ HYBRID SCORE  │
                 └───────────────┘
                         │
                Top K / Relevant chunks
                         ▼
    ┌────────────────────────────────────────┐
    │            RE-RANKER                   │   ← Prompt => Top K
    │  (cross-encoder / LLM re-scores)       │      LLM re-ranks the order
    └────────────────────────────────────────┘      based on the query
                         │
                Reordering => Rank
                 1,2,3,4,5 → 5,2,1,4,3
                         ▼
                    ┌─────────┐
                    │   LLM   │
                    └─────────┘
                         │
                         ▼
                Output => Accurate answer


## Stage by Stage

| Stage | Component | What it does |
|-------|-----------|--------------|
| **Input** | Query | User question enters the pipeline |
| **Path A — Exact** | **BM25** | Keyword matching (TF-IDF / BoW) on a **sparse** representation → returns Top K |
| **Path B — Similar** | **Embeddings + FAISS** | Semantic search via cosine similarity on **dense** vectors → returns Top K |
| **Fusion** | **Hybrid Score** | Blends both score lists by weight (α) → combined Top K of relevant chunks |
| **Refinement** | **Re-Ranker** | Cross-encoder / LLM re-scores each **query-doc pair** and reorders them |
| **Generation** | **LLM** | Generates the final answer from the reranked context |
| **Output** | Answer | Accurate, well-grounded response |

## Key Points from the Diagram

- **Two parallel paths:** the query goes to BM25 *and* to the embedding/FAISS side independently — BM25 has its own keyword index, it does not read the vector store.
- **Sparse vs Dense:** BM25 operates on a **sparse matrix** (keywords), FAISS on **dense vectors** (meaning). Each *outputs* a ranked Top-K with scores.
- **Hybrid score** = weighted blend of both score lists (the α knob).
- **The reorder is the whole point:** retriever order `1,2,3,4,5` becomes `5,2,1,4,3` after re-ranking — the doc originally ranked 5th was actually the most relevant.
- **Re-ranker ≠ final LLM:** the re-ranker is a separate scoring stage (cross-encoder reads query+doc *together*); the final LLM only generates the answer.



# MMR — Maximal Marginal Relevance (Hybrid Search Strategies)

## The problem it solves
Your retriever returns top-k by similarity... and the top 3 all say basically the **same thing**. You wasted your context window on duplicates.

Notes example — query "Langchain GenAI" returns:

All relevant, but repetitive. You learn nothing new from doc 3.

## What MMR does
Picks docs that are **both**:
1. **Relevant** to the query ✓
2. **Diverse** from each other (non-redundant) ✓

> It stops the retriever from returning near-identical docs that repeat the same content.

## The formula

MMR(d) = λ × Sim(d, q) − (1−λ) × max Sim(d, s)

- `Sim(d, q)` = how relevant the doc is to the **query** → *reward*
- `max Sim(d, s)` = how similar it is to **already-selected** docs → *penalty*
- `λ` (0 to 1) = the dial: **high λ → relevance**, **low λ → diversity**

So: **score = relevance minus redundancy.** Every candidate gets punished for looking like something you already picked.

## The worked example (λ = 0.7)

**Step 1** — pick the most relevant doc outright:

| Doc | Sim to query |
|-----|--------------|
| **D1** | 0.95 ✅ picked |
| D2  | 0.93 |
| D3  | 0.80 |

**Step 2** — now score D2 and D3 *with the redundancy penalty*:

Sim(D1,D2) = 0.90 ← very redundant

Sim(D1,D3) = 0.30 ← diverse


MMR(D2) = 0.7×0.93 − 0.3×0.90 = 0.651 − 0.27 = 0.381

MMR(D3) = 0.7×0.80 − 0.3×0.30 = 0.560 − 0.09 = 0.470 ✅ wins


**D3 wins despite being *less* relevant** (0.80 < 0.93), because D2 was nearly a duplicate of D1.

**Final order:** `D1, D3`

| Rank | Document | Reason |
|------|----------|--------|
| 1 | D1 | Highest relevance |
| 2 | D3 | Best diversity + relevance (MMR) |

## When to use MMR
- **RAG** — avoid feeding the LLM redundant documents
- **Chatbots** — FAQ, search apps, document browsers
- When your retriever **already returns many results** and you want coverage
- Pairs great with **hybrid retrieval** (dense + sparse)

→ Result is a **richer, more useful context** for the LLM.

## When NOT to use MMR

| Scenario | Why you might skip MMR |
|----------|------------------------|
| Extremely short context window | You may just want top-1 most relevant |
| You need **precision only** | Not focused on coverage |
| Documents are already diverse | No need to enforce diversity |
| You're already **reranking with an LLM** | Redundancy handled by post-filtering |

## TL;DR
> **Reranker** = "put the most relevant on top."
> **MMR** = "don't give me five copies of the same answer."
> `MMR = λ·relevance − (1−λ)·redundancy`, tune λ for how much you care about variety.

## In LangChain
```python
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.7}   # ← your λ
)